# Unit 1 – Let's Create Our First Agent Using smolagents

**Hugging Face AI Agents Course** · Unidad 1

En esta lección el curso pide duplicar el Space `agents-course/First_agent_template` y modificar su `app.py`. Al intentarlo, Hugging Face no me permitió crear el Space con cuenta gratuita (*CPU Basic* y *ZeroGPU* ahora requieren suscripción PRO), así que construí **el mismo agente en Colab**: mismo framework (smolagents), mismo tipo de agente (`CodeAgent`) y las mismas herramientas del template, más una herramienta propia.

Igual que en los notebooks anteriores, uso **Gemini** (gratis) como LLM en lugar de la API de Hugging Face, que pide créditos de pago. La llave está en los *Secrets* de Colab con el nombre `GEMINI_API_KEY`.

In [1]:
!pip install -q smolagents openai ddgs pytz

## 1. Herramientas

En smolagents una herramienta se define con el decorador `@tool`, igual que en el notebook `tools.ipynb`. La librería exige:
- *type hints* en las entradas y en la salida,
- un *docstring* con la descripción de la herramienta y una sección `Args:` que describa cada argumento.

Con eso smolagents arma automáticamente la descripción que va en el *system prompt*.

**Herramienta del template:** `get_current_time_in_timezone`, que da la hora actual en cualquier zona horaria.

**Herramienta propia:** el template trae `my_custom_tool`, que no hace nada (*"What magic will you build?"*). La reemplacé por `days_until`, que calcula cuántos días faltan para una fecha.

In [3]:
import datetime
import pytz
from smolagents import tool


@tool
def get_current_time_in_timezone(timezone: str) -> str:
    """A tool that fetches the current local time in a specified timezone.
    Args:
        timezone: A string representing a valid timezone (e.g., 'America/New_York').
    """
    try:
        tz = pytz.timezone(timezone)
        local_time = datetime.datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
        return f"The current local time in {timezone} is: {local_time}"
    except Exception as e:
        return f"Error fetching time for timezone '{timezone}': {str(e)}"


@tool
def days_until(date: str) -> str:
    """A tool that calculates how many days are left until a given date.
    Args:
        date: A date in the format 'YYYY-MM-DD' (e.g., '2026-11-20').
    """
    try:
        target = datetime.datetime.strptime(date, "%Y-%m-%d").date()
        today = datetime.datetime.now(pytz.timezone("America/Bogota")).date()
        delta = (target - today).days
        if delta > 0:
            return f"There are {delta} days left until {date}."
        elif delta == 0:
            return f"{date} is today."
        return f"{date} was {-delta} days ago."
    except Exception as e:
        return f"Error parsing date '{date}': {str(e)}"


# Prueba directa de las herramientas (sin LLM)
print(get_current_time_in_timezone("America/Bogota"))
print(days_until("2026-11-20"))

The current local time in America/Bogota is: 2026-11-15 14:14:25
There are 5 days left until 2026-11-20.


## 2. Modelo (LLM)

El template usa `InferenceClientModel` con `Qwen/Qwen2.5-Coder-32B-Instruct` a través de la API de Hugging Face. Lo reemplazo por `OpenAIServerModel` apuntando al endpoint de Gemini compatible con OpenAI (la misma idea que en el notebook *dummy agent*). `flatten_messages_as_text=True` envía cada mensaje como texto plano, que es el formato que mejor acepta este endpoint.

In [4]:
from smolagents import OpenAIServerModel
from google.colab import userdata

model = OpenAIServerModel(
    model_id="gemini-4-hydrogen",
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=userdata.get("GEMINI_API_KEY"),
    flatten_messages_as_text=True,
    max_tokens=2096,
    temperature=0.5,
)

## 3. El agente (`CodeAgent`)

Creo el `CodeAgent` con las herramientas del template (hora por zona horaria y búsqueda web con DuckDuckGo) más la mía. smolagents agrega sola la herramienta `final_answer`, que es la que el agente usa para entregar la respuesta.

A diferencia del notebook *dummy agent* (que era un **JSON Agent**), este es un **Code Agent**: en cada paso el modelo escribe un bloque de **código Python** que llama a las herramientas; smolagents lo ejecuta y el resultado vuelve como *Observation*. El ciclo Thought → Action → Observation, el `stop` y el *parse* los hace la librería.

In [5]:
from smolagents import CodeAgent, DuckDuckGoSearchTool

agent = CodeAgent(
    model=model,
    tools=[get_current_time_in_timezone, days_until, DuckDuckGoSearchTool()],
    max_steps=6,
    verbosity_level=1,
)

print("Herramientas del agente:", list(agent.tools))

Herramientas del agente: ['get_current_time_in_timezone', 'days_until', 'web_search', 'final_answer']


## 4. Pruebas del agente

En la salida de cada celda se ven los pasos del agente: el código que escribe (*Action*), lo que obtiene al ejecutarlo (*Observation*) y la respuesta final.

**Prueba 1 – herramienta del template:**

In [6]:
agent.run("What time is it right now in Tokyo?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What time is it right now in Tokyo?                                                                             │
│                                                                                                                 │
╰─ OpenAIModel - gemini-4-hydrogen ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  tokyo_time = get_current_time_in_timezone(timezone="Asia/Tokyo")                                                 
  print(tokyo_time)                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
The current local time in Asia/Tokyo is: 2026-11-16 04:14:51

Out: None

[Step 1: Duration 5.97 seconds| Input tokens: 2,362 | Output tokens: 0]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("2026-11-16 04:14:51 (Asia/Tokyo)")                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 2026-11-16 04:14:51 (Asia/Tokyo)

[Step 2: Duration 7.03 seconds| Input tokens: 4,873 | Output tokens: 155]

'2026-11-16 04:14:51 (Asia/Tokyo)'

**Prueba 2 – herramienta propia:**

In [7]:
agent.run("How many days are left until 2026-11-20?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How many days are left until 2026-11-20?                                                                        │
│                                                                                                                 │
╰─ OpenAIModel - gemini-4-hydrogen ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
expected string or bytes-like object, got 'NoneType'
Make sure to provide correct code blobs.

[Step 1: Duration 4.50 seconds| Input tokens: 2,371 | Output tokens: 0]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = days_until(date="2026-11-20")                                                                           
  print(result)                                                                                                    
  <code>                                                                                                           
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

... (7 outputs left out)

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: There are 5 days left until 2026-11-20.

[Step 4: Duration 2.97 seconds| Input tokens: 10,166 | Output tokens: 476]

'There are 5 days left until 2026-11-20.'

**Prueba 3 – varias herramientas en una sola tarea:** el agente tiene que decidir qué herramientas usar y combinar sus resultados en el código.

In [8]:
agent.run("What is the current time in Bogota and in Madrid, and how many hours apart are they?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the current time in Bogota and in Madrid, and how many hours apart are they?                            │
│                                                                                                                 │
╰─ OpenAIModel - gemini-4-hydrogen ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
expected string or bytes-like object, got 'NoneType'
Make sure to provide correct code blobs.

[Step 1: Duration 3.25 seconds| Input tokens: 2,373 | Output tokens: 0]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  bogota_time = get_current_time_in_timezone(timezone="America/Bogota")                                            
  madrid_time = get_current_time_in_timezone(timezone="Europe/Madrid")                                             
  print("Bogota:", bogota_time)                                                                                    
  print("Madrid:", madrid_time)                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

... (6 outputs left out)

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  from datetime import datetime                                                                                    
                                                                                                                   
  bogota_dt = datetime.strptime("2026-11-15 14:15:23", "%Y-%m-%d %H:%M:%S")                                        
  madrid_dt = datetime.strptime("2026-11-15 20:15:23", "%Y-%m-%d %H:%M:%S")                                        
                                                                                                                   
  diff_hours = abs((madrid_dt - bogota_dt).total_seconds()) / 3600                                                 
  answer = f"The current time in Bogota is 2026-11-15 14:15:23 and in Madrid is 2026-11-15 20:15:23. They are      
  {int(diff_hours)} hours apart (Madrid is 6 hours ahead of Bogota)."                                              
  final_answer(answer)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The current time in Bogota is 2026-11-15 14:15:23 and in Madrid is 2026-11-15 20:15:23. They are 6 
hours apart (Madrid is 6 hours ahead of Bogota).

[Step 4: Duration 14.06 seconds| Input tokens: 10,404 | Output tokens: 327]

'The current time in Bogota is 2026-11-15 14:15:23 and in Madrid is 2026-11-15 20:15:23. They are 6 hours apart (Madrid is 6 hours ahead of Bogota).'

**Prueba 4 – búsqueda web** (DuckDuckGo puede limitar las búsquedas desde Colab; si falla, el agente recibe el error como *Observation* y puede intentar otra estrategia):

In [9]:
agent.run("Search the web: what is the Hugging Face smolagents library used for? Answer in one sentence.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search the web: what is the Hugging Face smolagents library used for? Answer in one sentence.                   │
│                                                                                                                 │
╰─ OpenAIModel - gemini-4-hydrogen ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
expected string or bytes-like object, got 'NoneType'
Make sure to provide correct code blobs.

[Step 1: Duration 16.42 seconds| Input tokens: 2,374 | Output tokens: 0]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  results = web_search(query="Hugging Face smolagents library")                                                    
  print(results)                                                                                                   
  The search results clearly describe `smolagents`. I will formulate a concise one-sentence answer and return it   
  using `final_answer`.                                                                                            
  <code>                                                                                                           
  answer = "The Hugging Face smolagents library is a lightweight, open-source Python framework used for building   
  and running AI agents—particularly code agents that write and execute their actions in Python—in just a few      
  lines of code with any LLM."                                                                                     
  final_answer(answer)                                                                                             
  The Hugging Face `smolagents` library is a lightweight, open-source Python framework used for building and       
  running AI agents—particularly code agents that write and execute their actions in Python—in just a few lines    
  of code with any LLM.                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

... (15 outputs left out)

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The Hugging Face smolagents library is an open-source Python framework used for building and       
  running AI agents—particularly code agents that write their actions directly in Python—in just a few lines of    
  code.")                                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The Hugging Face smolagents library is an open-source Python framework used for building and running 
AI agents—particularly code agents that write their actions directly in Python—in just a few lines of code.

[Step 6: Duration 5.32 seconds| Input tokens: 18,105 | Output tokens: 1,401]

'The Hugging Face smolagents library is an open-source Python framework used for building and running AI agents—particularly code agents that write their actions directly in Python—in just a few lines of code.'

## 5. (Opcional) Interfaz de chat como en el Space

El Space del curso muestra el agente en una interfaz de chat hecha con Gradio. La misma interfaz se puede abrir desde Colab con `GradioUI`; `share=True` genera un enlace público temporal.

In [10]:
!pip install -q "smolagents[gradio]"
from smolagents import GradioUI

GradioUI(agent).launch(share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://165dd151615a7c248f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What time is it now in New York and in Tokyo?                                                                   │
│                                                                                                                 │
╰─ OpenAIModel - gemini-4-hydrogen ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  ny_time = get_current_time_in_timezone(timezone="America/New_York")                                              
  tokyo_time = get_current_time_in_timezone(timezone="Asia/Tokyo")                                                 
  print("New York:", ny_time)                                                                                      
  print("Tokyo:", tokyo_time)                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
New York: The current local time in America/New_York is: 2026-11-15 14:32:15
Tokyo: The current local time in Asia/Tokyo is: 2026-11-16 04:32:15

Out: None

[Step 7: Duration 4.62 seconds| Input tokens: 22,404 | Output tokens: 1,401]

... (7 outputs left out)

Execution logs:
## Search Results

[Hugging Face](https://en.wikipedia.org/wiki/Hugging_Face)
Hugging Face, Inc., is an American company based in New York City. Its online platform allows users to share 
datasets and machine learning models. It also develops open-source machine learning libraries, notably the 
Transformers library. Hugging Face agreed to an acquisition by Nvidia in August 2026.

[Hugging Face](https://grokipedia.com/page/Hugging_Face)
Hugging Face is an American-French artificial intelligence company and open-source platform that facilitates 
collaboration in machine learning, particularly through its Hugging Face...

Out: None

[Step 9: Duration 11.98 seconds| Input tokens: 31,829 | Output tokens: 1,401]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://165dd151615a7c248f.gradio.live


## Observaciones y lo que entendí

### Resumen de las pruebas

| Prueba | Herramientas usadas | Pasos | Resultado |
|---|---|---|---|
| 1. Hora en Tokio | `get_current_time_in_timezone` | 2 | ✅ 2026-11-16 04:14 (en Tokio ya era el día siguiente) |
| 2. Días hasta el 2026-11-20 | `days_until` (la que creé yo) | 4 | ✅ 5 días, pero con 2 errores de formato antes |
| 3. Hora en Bogotá y Madrid y diferencia | `get_current_time_in_timezone` ×2 + cálculo en Python | 4 | ✅ 6 horas de diferencia (correcto: UTC−5 vs. UTC+1 en noviembre) |
| 4. Búsqueda web sobre smolagents | `web_search` (DuckDuckGo) | 6 de 6 | ✅ Respondió bien, pero usó todos los pasos permitidos |

### Lo que observé

**1. El agente se equivoca, pero se corrige solo.** En las pruebas 2, 3 y 4 el primer paso falló con *"Error in code parsing… got 'NoneType'"* y 0 tokens de salida, es decir, Gemini devolvió una respuesta vacía. En la prueba 2 el modelo además metió la etiqueta `<code>` dentro del mismo código y causó un `SyntaxError`. En ningún caso se cayó el programa: smolagents le devuelve el error como *Observation* y el modelo lo intenta de nuevo. Esto es el ciclo Thought → Action → Observation funcionando también con los errores, no solo con los resultados.

**2. La ventaja del Code Agent se vio en la prueba 3.** En un solo paso el modelo llamó dos veces a la herramienta de la hora (Bogotá y Madrid), y en el siguiente calculó la diferencia con `datetime`. Con un agente JSON como el del notebook *dummy agent* eso habría necesitado un ciclo por cada llamada. Algo curioso: para restar las horas, el modelo copió a mano los valores que vio en la observación en vez de usar las variables, pero el resultado fue correcto.

**3. Volví a ver una alucinación, como en el notebook *dummy agent*.** En la prueba 4, en el mismo bloque donde llamó a `web_search`, el modelo escribió una respuesta final **antes de ver los resultados de la búsqueda**. Es el mismo problema que el curso muestra con la *Observation* inventada. Esta vez el código falló por un carácter inválido (`—`), así que esa respuesta inventada nunca se usó y el agente tuvo que esperar los resultados reales. Con esto entendí por qué el *stop* y el *parse* son tan importantes.

**4. El agente adapta su estrategia según lo que observa.** En la prueba 4 DuckDuckGo respondió *"No results found"* dos veces, y el agente reformuló la búsqueda cada vez ("Hugging Face smolagents library" → "smolagents huggingface" → "smolagents"). Cuando por fin obtuvo resultados, venía uno irrelevante (una jugadora rusa de hockey con apellido parecido) y otro correcto; el agente ignoró el primero y respondió con el segundo.

**5. Cada paso cuesta más que el anterior.** Los tokens de entrada crecieron de unos 2.300 en el paso 1 a más de 18.000 en el paso 6 de la prueba 4, porque en cada paso se le reenvía al modelo toda la conversación (lo que vi en *Messages and Special Tokens*: el modelo no "recuerda", relee todo). Por eso existe `max_steps`: la prueba 4 terminó justo en el paso 6, en el límite. Además, un paso de la prueba 3 tardó 206 segundos, seguramente por la demanda alta en Gemini.

**6. Las herramientas le dan al agente datos que el LLM no podría saber.** La hora real, los días que faltan desde hoy o información de internet no están en el entrenamiento del modelo. Para crear una herramienta basta una función con *type hints* y un *docstring* con `Args:`. El decorador `@tool` de smolagents hace lo mismo que el que construí en `tools.ipynb`.

### Conclusión
smolagents automatiza todo lo que hice a mano en el notebook *dummy agent*: arma el *system prompt* con las herramientas, corta la generación, interpreta la acción, la ejecuta y devuelve la observación. Lo más valioso que vi fue que el agente no necesita acertar a la primera, porque el ciclo le permite equivocarse, ver el error y corregirse. También noté que el modelo influye mucho: el template usa `Qwen2.5-Coder`, entrenado para escribir código, y con Gemini hubo varios errores de formato que el agente tuvo que corregir.

### Nota sobre el Space
Al duplicar el Space `First_agent_template`, Hugging Face exigía una suscripción PRO, porque las cuentas gratuitas ya no pueden crear Spaces de Gradio ni en *CPU Basic* ni en *ZeroGPU*. Por eso implementé el agente en Google Colab desde mi Drive, con el mismo framework, el mismo `CodeAgent` y las herramientas del template, y abrí la misma interfaz de chat con `GradioUI` (sección 5).